In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
import nltk
from nltk.corpus import gutenberg

# 1. Download NLTK resources and load the text
nltk.download('gutenberg')
text = gutenberg.raw('shakespeare-hamlet.txt').lower() # Convert to lowercase for simplicity
print(f"Total characters in Hamlet: {len(text)}")
print("--- Sample Text ---")
print(text[:500])
print("-------------------")

[nltk_data] Downloading package gutenberg to /root/nltk_data...


Total characters in Hamlet: 162881
--- Sample Text ---
[the tragedie of hamlet by william shakespeare 1599]


actus primus. scoena prima.

enter barnardo and francisco two centinels.

  barnardo. who's there?
  fran. nay answer me: stand & vnfold
your selfe

   bar. long liue the king

   fran. barnardo?
  bar. he

   fran. you come most carefully vpon your houre

   bar. 'tis now strook twelue, get thee to bed francisco

   fran. for this releefe much thankes: 'tis bitter cold,
and i am sicke at heart

   barn. haue you had quiet guard?
  fran. not
-------------------


[nltk_data]   Unzipping corpora/gutenberg.zip.


In [ ]:
#Tokenization and Sequence Creation
# 2. Character Vocabulary & Encoding
chars = sorted(list(set(text)))
char_to_index = dict((c, i) for i, c in enumerate(chars))
index_to_char = dict((i, c) for i, c in enumerate(chars))
vocab_size = len(chars)

print(f"Total unique characters (Vocabulary Size): {vocab_size}")

# 3. Sequence Creation (Sliding Window)
SEQUENCE_LENGTH = 100 # Hyperparameter: The size of the context window
STEP = 3            # How many characters to skip between sequences

sequences = []
next_chars = []

for i in range(0, len(text) - SEQUENCE_LENGTH, STEP):
    # Input sequence (100 characters)
    sequences.append(text[i : i + SEQUENCE_LENGTH])
    # Target character (the 101st character)
    next_chars.append(text[i + SEQUENCE_LENGTH])

print(f"Total sequences created: {len(sequences)}")

# 4. Vectorization (One-Hot Encoding)
# X: Input (sequences), Y: Target (next characters)
X = np.zeros((len(sequences), SEQUENCE_LENGTH, vocab_size), dtype=bool)
y = np.zeros((len(sequences), vocab_size), dtype=bool)

for i, sequence in enumerate(sequences):
    for t, char in enumerate(sequence):
        # Set the corresponding index to True (1)
        X[i, t, char_to_index[char]] = 1
    # Set the target character index to True (1)
    y[i, char_to_index[next_chars[i]]] = 1

print(f"X shape (Input): {X.shape}")
print(f"y shape (Output): {y.shape}")

Total unique characters (Vocabulary Size): 44
Total sequences created: 54261
X shape (Input): (54261, 100, 44)
y shape (Output): (54261, 44)


In [ ]:


#3. Model Definition and Training
# 5. Build the LSTM Model (REVISED for Faster Convergence)

# Model Architecture Hyperparameters:
# Increase capacity to learn more with fewer passes
SEQUENCE_LENGTH = 100
LSTM_UNITS = 512 # Increased capacity (up from 128)

model = Sequential([
    LSTM(LSTM_UNITS, input_shape=(SEQUENCE_LENGTH, vocab_size)),
    # Add a Dropout layer before the final Dense layer for regularization
    tf.keras.layers.Dropout(0.2),
    Dense(vocab_size, activation='softmax')
])

# Training Hyperparameters:
# CRITICAL: Lowest possible stable Learning Rate for better convergence
LEARNING_RATE = 0.0005
# Reduced max epochs, relying on early stopping
EPOCHS = 40
BATCH_SIZE = 128

# Use the gentler Adam optimizer, which often converges faster than RMSprop at low LRs
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Early Stopping: Still essential to stop when the model hits its best performance
# Monitor 'loss' and stop if no improvement for a few epochs (patience=5)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='loss',
    patience=5,
    restore_best_weights=True
)

model.summary()

# 6. Train the Model
print("\n--- Model Training Started (Max 40 Epochs) ---")
history = model.fit(
    X,
    y,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.05,
    callbacks=[early_stopping]
)
print("--- Model Training Finished ---")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 512)            │     1,140,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 44)             │        22,572 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,163,308 (4.44 MB)

 Trainable params: 1,163,308 (4.44 MB)

 Non-trainable params: 0 (0.00 B)


--- Model Training Started (Max 40 Epochs) ---
Epoch 1/40
403/403 ━━━━━━━━━━━━━━━━━━━━ 966s 2s/step - accuracy: 0.1690 - loss: 3.0582 - val_accuracy: 0.2642 - val_loss: 2.5810
Epoch 2/40
403/403 ━━━━━━━━━━━━━━━━━━━━ 984s 2s/step - accuracy: 0.2825 - loss: 2.5165 - val_accuracy: 0.3382 - val_loss: 2.2863
Epoch 3/40
403/403 ━━━━━━━━━━━━━━━━━━━━ 968s 2s/step - accuracy: 0.3385 - loss: 2.2856 - val_accuracy: 0.3626 - val_loss: 2.1648
Epoch 4/40
403/403 ━━━━━━━━━━━━━━━━━━━━ 965s 2s/step - accuracy: 0.3717 - loss: 2.1647 - val_accuracy: 0.3913 - val_loss: 2.0769
Epoch 5/40
403/403 ━━━━━━━━━━━━━━━━━━━━ 956s 2s/step - accuracy: 0.3892 - loss: 2.0831 - val_accuracy: 0.3979 - val_loss: 2.0210
Epoch 6/40
403/403 ━━━━━━━━━━━━━━━━━━━━ 951s 2s/step - accuracy: 0.4106 - loss: 2.0041 - val_accuracy: 0.4186 - val_loss: 1.9635
Epoch 7/40
403/403 ━━━━━━━━━━━━━━━━━━━━ 949s 2s/step - accuracy: 0.4168 - loss: 1.9590 - val_accuracy: 0.4337 - val_loss: 1.9198
Epoch 8/40
403/403 ━━━━━━━━━━━━━━━━━━━━ 976s 2s/s

In [4]:
#Text Generation Function
# 7. Text Generation Function (Sampling with 'temperature')
def sample(preds, temperature=1.0):
    # Helper function to sample an index from a probability array
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

def generate_text(model, seed_text, generation_length=400, temperature=0.5):
    generated = ""
    sentence = seed_text.lower() # Start with the seed text
    generated += sentence

    # Loop to generate new characters
    for i in range(generation_length):
        # Prepare the input for the model: one-hot encoded and correct shape
        x_pred = np.zeros((1, SEQUENCE_LENGTH, vocab_size))
        for t, char in enumerate(sentence):
            x_pred[0, t, char_to_index[char]] = 1.

        # Predict the next character probabilities
        preds = model.predict(x_pred, verbose=0)[0]

        # Sample the next character using the temperature function
        next_index = sample(preds, temperature)
        next_char = index_to_char[next_index]

        # Update the sequence for the next prediction
        sentence = sentence[1:] + next_char
        generated += next_char

    return generated

# 8. Run Generation Example
print("\n--- Generated Text (Temperature=0.5 for coherence) ---")
SEED_TEXT = "to be or not to be, that is the question" # A famous line from Hamlet
generated_text = generate_text(model, SEED_TEXT, generation_length=400, temperature=0.5)
print(generated_text)
print("-------------------------------------------------------")


--- Generated Text (Temperature=0.5 for coherence) ---
to be or not to be, that is the questionooa,tytyyrtrmm?imwt yutrrai eyos a lyaraaa;c,irtoeaaaanwto,iloeitr t?ll,t,u atac,iilerlsus,cel,eieeryaryaaaaaaatratoarrlyi iar,alditcaiiisl,oeea tttsutwrteeea,e,trinaiacorliuaaa
eaettdiiw,baryyii,iiir, i?ia?itt,ot wll,i,ui,yairaei,ifryyssaiatdntai,yyir,rt,t,ttibeerytaaai,a iaaeurrieoeaii t
ilootay,ubyyl,iada,eatea,,t,sm,utui imytriomi
iotdraaeaaia,uoie,ieityi,,yriceiie'eifiustera,eireopyty
twtaydl
-------------------------------------------------------
